## 6.2 图像卷积
### 6.2.1 互相关运算
把卷积核在图像上的每一个像素进行移动，然后对应位置计算乘积，最后求和。

In [6]:
import torch
from torch import nn

def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

### 6.2.2 卷积层
卷积层对输入和卷积核权重进行互相关运算，并在添加标量偏置之后产生输出。
因此卷积层的两个被训练的参数是**卷积核权重和偏置**。

In [7]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))
        
    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

### 6.2.3 图像中目标的边缘检测
边缘检测，即找出图像中边界的像素点。
**一个简单的应用：**通过找到像素变化的位置，来检测图像中不同颜色的边缘。

In [8]:
# 构造一个6 * 8像素的黑白图像。中间四列为黑色，其余像素为白色。
X = torch.ones((6 ,8))
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [9]:
# 一个1 * 2 的卷积核K，如果两水平相邻元素相同，则输出为零，否则非零
K = torch.tensor([[1.0, -1.0]])
Y = corr2d(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [10]:
corr2d(X.t(), K)  # 说明K只能检测垂直边缘，无法检测水平边缘

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

### 6.2.4 学习卷积核
尝试仅通过查看“输入-输出”对来学习由X生成Y的卷积核。

In [14]:
# 构造一个二维卷积层，它具有1个输出通道和形状为（1，2）的卷积核
corr2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)

# 这个二维卷积层使用四维输入和输出格式（批量大小、通道、高度、宽度），
# 其中批量大小和通道数都为1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2

for i in range(10):
    Y_hat = corr2d(X)
    l = (Y_hat - Y) ** 2
    corr2d.zero_grad()
    l.sum().backward()
    corr2d.weight.data -= lr * corr2d.weight.grad
    if(i + 1) % 2 == 0:
        print(f'batch {i + 1}, loss {l.sum():.3f}')
        
corr2d.weight.data.reshape((1, 2))

batch 2, loss 13.670
batch 4, loss 3.456
batch 6, loss 1.056
batch 8, loss 0.372
batch 10, loss 0.142


tensor([[ 0.9465, -1.0225]])